# AutoDine A：YOLO26 摄像头实时视觉原型

本 Notebook 使用真实 YOLO26 模型完成**当前视野**中的苹果、香蕉、橙子和人员检测，并对检测到的水果执行整果 `good / defective / review` 分类。它不会累计同一物体经过多少次，也不会定位水果表面的病斑或破损区域。

门磁、门锁和授权信息尚未接入，因此这里只显示当前视野人数，不判断未授权进入。没有使用 Mock。

In [ ]:
from pathlib import Path
import sys

def find_repo_root():
    candidates = [Path.cwd(), Path.cwd() / 'AutoDine', *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'edge' / 'smart_storage_vision' / 'src').is_dir():
            return candidate.resolve()
    raise RuntimeError('找不到 AutoDine 根目录；请从解压后的 AutoDine 目录启动 JupyterLab')

REPO_ROOT = find_repo_root()
MODULE_ROOT = REPO_ROOT / 'edge' / 'smart_storage_vision'
sys.path.insert(0, str(MODULE_ROOT / 'src'))

DETECTOR_PATH = REPO_ROOT / 'yolo26n.pt'
QUALITY_MODEL_PATH = MODULE_ROOT / 'models' / 'fruit_quality_yolo26_best.pt'
for path in (DETECTOR_PATH, QUALITY_MODEL_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)
print('AutoDine:', REPO_ROOT)
print('检测模型:', DETECTOR_PATH)
print('品质分类模型:', QUALITY_MODEL_PATH)

In [ ]:
import cv2
import torch
import ultralytics

print('OpenCV:', cv2.__version__)
print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
if ultralytics.__version__ != '8.4.123':
    raise RuntimeError('本原型要求 ultralytics==8.4.123；旧版可能加载YOLO26权重但返回错误检测结果')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 摄像头来源

USB 摄像头通常从 `0` 开始；如果打不开可依次尝试 `1`、`2`。RTSP 摄像头可填写完整地址，但不要把含密码的地址保存或上传。

In [ ]:
CAMERA_SOURCE = 0  # USB: 0/1/2；RTSP: 'rtsp://...'

## 启动实时推理

运行后单元格会很快结束，推理在线程中继续，Jupyter 内会持续刷新画面。点击红色“停止摄像头”按钮即可释放摄像头。计数是当前画面中的数量，不是累计通过数量。

In [ ]:
from smart_storage_vision.jupyter_camera import JupyterCameraSession

session = JupyterCameraSession(
    detector_path=DETECTOR_PATH,
    quality_model_path=QUALITY_MODEL_PATH,
    camera_source=CAMERA_SOURCE,
    detection_confidence=0.4,
    quality_confidence=0.7,
    display_width=960,
)
session.show()
session.start()

## 手动停止或检查后台错误

通常直接点击画面下方按钮。也可以运行下面的单元格。若画面显示运行失败，停止后再次运行本单元格会把原始异常抛出，方便诊断，不会吞掉错误。

In [ ]:
session.stop()
if session.error is not None:
    raise session.error